In [ ]:
# Machine Learning Pipeline for Beryllium-Free Vitreloy 1 Replacements
# This script executes data preprocessing, classification, threshold optimization,
# high-throughput combinatorial screening (via matminer), and visualization.

!pip install -q matminer pymatgen

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty

# ==============================================================================
# PHASE 1 & 2: Data Loading and Preprocessing
# ==============================================================================
print("Loading and preprocessing data...")

# Load dataset
df = pd.read_csv('final_dataset.csv')

# Clean citation brackets from Alloy names (e.g., "Mg70Ni15Nd15[24]" -> "Mg70Ni15Nd15")
if 'Alloy' in df.columns:
    df['Alloy'] = df['Alloy'].str.replace(r'\[\d+\]', '', regex=True)

# Identify feature columns (X1 through X17) and target column
feature_cols = [col for col in df.columns if re.match(r'^X\d+', col)]
target_col = 'Y (Dmax)'

# Drop rows with missing values in features or target
cols_to_check = feature_cols + [target_col]
df_clean = df.dropna(subset=cols_to_check).copy()

X = df_clean[feature_cols]
y = df_clean[target_col]

# ==============================================================================
# PHASE 3: Classification Pivot (Predicting Exceptional Glass Formers)
# ==============================================================================
print("\nTraining Baseline Classifier...")

# Convert Dmax into a Binary Target (0 = Standard, 1 = Exceptional)
threshold = 10.0
y_binary = (y >= threshold).astype(int)

# Train/Test Split (Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

# Train the Classifier
rf_classifier = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1
)
rf_classifier.fit(X_train, y_train)

# Predict and Evaluate
y_pred = rf_classifier.predict(X_test)
y_prob = rf_classifier.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_prob)

print(classification_report(y_test, y_pred, target_names=['Standard (<10mm)', 'Exceptional (>=10mm)']))
print(f"ROC-AUC Score: {roc_auc:.4f}")

# Plot Baseline Confusion Matrix and ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Standard', 'Exceptional'], yticklabels=['Standard', 'Exceptional'])
axes[0].set_title('Baseline Confusion Matrix')
axes[0].set_ylabel('Actual Class')
axes[0].set_xlabel('Predicted Class')

RocCurveDisplay.from_estimator(rf_classifier, X_test, y_test, ax=axes[1], color='darkorange')
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].set_title('ROC Curve')
plt.tight_layout()
plt.show()

# Plot Feature Importance
importances = rf_classifier.feature_importances_
indices = np.argsort(importances)[-10:]
plt.figure(figsize=(8, 5))
plt.barh(range(10), importances[indices], align='center')
plt.yticks(range(10), [X.columns[i] for i in indices])
plt.xlabel('Relative Importance')
plt.title('Top 10 Features for Predicting Exceptional BMGs')
plt.tight_layout()
plt.show()

# ==============================================================================
# PHASE 4: Threshold Optimization for Experimental Screening
# ==============================================================================
print("\nOptimizing Decision Threshold...")

# Lower threshold to 0.30 to prioritize recall (finding champions)
custom_threshold = 0.30
y_pred_adjusted = (y_prob >= custom_threshold).astype(int)

print(classification_report(y_test, y_pred_adjusted, target_names=['Standard', 'Exceptional']))

plt.figure(figsize=(6, 5))
cm_adjusted = confusion_matrix(y_test, y_pred_adjusted)
sns.heatmap(cm_adjusted, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Standard', 'Exceptional'], yticklabels=['Standard', 'Exceptional'])
plt.title(f'Adjusted Confusion Matrix (Threshold > {custom_threshold})')
plt.ylabel('Actual Class')
plt.xlabel('Predicted Class')
plt.tight_layout()
plt.show()

# ==============================================================================
# PHASE 5: High-Throughput Screening (Matminer)
# ==============================================================================
print("\nStarting High-Throughput Screening with Matminer...")

# Initialize Magpie featurizer
ep = ElementProperty.from_preset("magpie")
feature_names = ep.feature_labels()

# Parse original alloys to Pymatgen objects
valid_indices = []
comp_objects = []
for idx, alloy_str in df_clean['Alloy'].items():
    try:
        comp = Composition(alloy_str)
        comp_objects.append(comp)
        valid_indices.append(idx)
    except:
        pass

df_matminer = df_clean.loc[valid_indices].copy()
y_binary_matminer = y_binary.loc[valid_indices]
df_matminer['comp_obj'] = comp_objects

# Featurize training data
df_matminer = ep.featurize_dataframe(df_matminer, col_id='comp_obj', ignore_errors=True)
X_train_magpie = df_matminer[feature_names]

# Retrain model on Magpie features
rf_magpie = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
rf_magpie.fit(X_train_magpie, y_binary_matminer)

# Generate Search Space: Zr41.2 Ti13.8 Cu12.5 Ni10 (Al, Ag, Si substitution for 22.5 Be)
base_str = "Zr41.2 Ti13.8 Cu12.5 Ni10 "
new_alloys = []
step = 0.5

for al in np.arange(0.0, 25.0, step):
    for ag in np.arange(0.0, 25.0, step):
        si = 22.5 - al - ag
        if si >= 0.0 and np.isclose(al + ag + si, 22.5):
            comp_str = base_str
            if al > 0: comp_str += f"Al{al:.1f} "
            if ag > 0: comp_str += f"Ag{ag:.1f} "
            if si > 0: comp_str += f"Si{si:.1f}"
            new_alloys.append(comp_str.strip())

# Featurize and Predict new alloys
df_search = pd.DataFrame({'Alloy': new_alloys})
df_search['comp_obj'] = df_search['Alloy'].apply(lambda x: Composition(x))
df_search = ep.featurize_dataframe(df_search, col_id='comp_obj')
X_search = df_search[feature_names]

df_search['Prob_Exceptional'] = rf_magpie.predict_proba(X_search)[:, 1]

# Output Top 15
df_results = df_search[['Alloy', 'Prob_Exceptional']].sort_values(by='Prob_Exceptional', ascending=False).reset_index(drop=True)
print("\nTOP 15 RECOMMENDED BE-FREE VITRELOY 1 REPLACEMENTS:")
for i in range(15):
    print(f"{i+1:2d}. {df_results.loc[i, 'Alloy']:<40} | Confidence: {df_results.loc[i, 'Prob_Exceptional']*100:.1f}%")

# ==============================================================================
# PHASE 6: Visualizing the Discovery Landscape
# ==============================================================================
print("\nGenerating Discovery Heatmap...")

# Parse Al, Ag, Si values for heatmap
def parse_substitutions(alloy_string):
    composition = {}
    matches = re.findall(r'(Al|Ag|Si)(\d+\.?\d*)', alloy_string)
    for element, value in matches:
        composition[element] = float(value)
    return composition

parsed_data = df_search['Alloy'].apply(parse_substitutions)
df_search['Al'] = parsed_data.apply(lambda x: x.get('Al', 0))
df_search['Ag'] = parsed_data.apply(lambda x: x.get('Ag', 0))

# Create pivot table and fill NaNs
pivot_table = df_search.pivot_table(index='Ag', columns='Al', values='Prob_Exceptional')
pivot_table = pivot_table.fillna(pivot_table.min().min())

# Plot Heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pivot_table, ax=ax, cmap="magma", annot=False, cbar_kws={'label': 'Predicted Probability'})

# Annotate Best Candidate
best_row = df_search.loc[df_search['Prob_Exceptional'].idxmax()]
y_pos = pivot_table.index.get_loc(best_row['Ag'])
x_pos = pivot_table.columns.get_loc(best_row['Al'])
ax.text(x_pos + 0.5, y_pos + 0.5, '*', ha='center', va='center', color='white', fontsize=30)

ax.set_title('Predicted GFA in the Al-Ag Substitution Space')
ax.set_xlabel('Aluminum Content (at.%)')
ax.set_ylabel('Silver Content (at.%)')
ax.invert_yaxis()

plt.tight_layout()
plt.show()